# Trabalho de DL - Tradutor (Transformer) - Versão usando Pytorch - G13

### 1. Importando as bibliotecas

In [255]:
import random

from datasets import load_dataset

import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F

import math

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

### 2. Escolhendo o Device

In [256]:
# Verificar se temos GPU disponível
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### 3. Hyperparâmetros

In [257]:
# Hyperparametros do trabalho
NUM_SAMPLES = 50000     # Número de amostras a serem usadas para treinamento do tokenizador
BATCH_SIZE = 32         # Tamanho de cada mini batch
#VOCAB_SIZE = 16000     # Tamanho do vocabulário do Tokenizador
MAX_TEXT_LENGTH = 64    # Limita o tamanho máximo do texto
D_MODEL = 128           # Dimensões do modelo
NUM_HEADS = 8           # Número de cabeças do mecanismo de atenção
NUM_ENCODER_LAYERS = 4  # Número de camadas de encoder e decoder
D_FF = 256              # Dimensões do Feed Forward
DROPOUT = 0.1           # Taxa de dropout

### 4. Carregando os dados

In [258]:
dataset = load_dataset(
    "Helsinki-NLP/opus-100",
    "en-pt"
)

In [259]:
print(dataset)

In [260]:
# Um milhão de amostras é muita coisa, reduzindo o treino para 50.000


#train = dataset["train"].select(range(NUM_SAMPLES)) # As 50.000 primeiras

train = dataset["train"].shuffle(seed=13).select(range(NUM_SAMPLES)) # Escolha aleatória

valid = dataset["validation"]

test = dataset["test"]

In [261]:
print(train[0]) # primeira (1)
print(train[-1]) # última (50.000)


In [262]:
# Escolha 5 amostras da base de treino
for i in random.sample(range(NUM_SAMPLES), 5):

    exemplo = train[i]["translation"]

    print("Português :", exemplo["pt"])
    print("Inglês    :", exemplo["en"])
    print("-"*60)

In [263]:
# Separando os idiomas
train_pt = [
    exemplo["translation"]["pt"]
    for exemplo in train
]

train_en = [
    exemplo["translation"]["en"]
    for exemplo in train
]

In [264]:
valid_pt = [
    exemplo["translation"]["pt"]
    for exemplo in valid
]

valid_en = [
    exemplo["translation"]["en"]
    for exemplo in valid
]

test_pt = [
    exemplo["translation"]["pt"]
    for exemplo in test
]

test_en = [
    exemplo["translation"]["en"]
    for exemplo in test
]

In [265]:
print(train_pt[0])
print(train_en[0])

In [266]:
print(f"Treino    : {len(train_pt)}")
print(f"Validação : {len(valid_pt)}")
print(f"Teste     : {len(test_pt)}")

### 5. Tokenização

In [267]:
from transformers import AutoTokenizer


In [268]:
# Carrega o BPE pré-treinado do Llama 3
#tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B")
#tokenizer = AutoTokenizer.from_pretrained("NousResearch/Meta-Llama-3-8B-Alternate-Tokenizer")
#tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
tokenizer = AutoTokenizer.from_pretrained("t5-small")

#VOCAB_SIZE = tokenizer.vocab_size
VOCAB_SIZE = len(tokenizer) # Inclui os tokens especiais
print("Tamanho do vocabulário é :", VOCAB_SIZE)

In [ ]:
# Função que converte a setença em tokens numéricos (ids)
def encoding(sentence):
    return tokenizer.encode(
        sentence,
        truncation=True,
        padding=False,
        add_special_tokens=True,
        max_length=MAX_TEXT_LENGTH)


In [ ]:
# Função que converte os ids de volta ao texto
def decoding(ids):
    return tokenizer.decode(ids,
        skip_special_tokens=True)

In [ ]:
# Teste
ids_pt = encoding("O aprendizado de máquina é incrível.")
print("PT:", ids_pt)
print(decoding(ids_pt))

In [ ]:
# Teste
ids_en = encoding("Machine learning is amazing.")
print("EN:", ids_en)
print(decoding(ids_en))

In [ ]:
# Retorna uma lista com as strings dos tokens especiais principais
print(tokenizer.all_special_tokens)
# Exemplo de saída: ['<|begin_of_text|>', '<|end_of_text|>', '<|reserved_special_token_0|>', ...]

# Retorna um dicionário mostrando a função de cada um e seu respectivo ID numérico
print(tokenizer.special_tokens_map)
print(tokenizer.all_special_ids)

O tokenizador utilizado foi baseado no t5-small

In [ ]:
# bos_id = tokenizer.convert_tokens_to_ids('<|begin_of_text|>')
# print(bos_id)

In [ ]:
eos_id = tokenizer.convert_tokens_to_ids('</s>')
print(eos_id)

In [ ]:
print(tokenizer.pad_token)
pad_id = tokenizer.pad_token_id
print(pad_id)


In [ ]:
# teste de lote
frases = [
    "bom dia",
    "eu gosto de programação",
    "o transformer utiliza atenção",
    "como você está"
]

for frase in frases:
    ids = encoding(frase)

    reconstruida = decoding(ids)

    print("-" * 50)
    print("Original     :", frase)
    print("Reconstruída :", reconstruida)

In [ ]:
frase = "O transformador é eficiente"

#ids = tokenizer.encode(frase_com_eos)
ids = encoding(frase)

print("Frase :", frase)
print("Tokens:", tokenizer.convert_ids_to_tokens(ids))
print("IDs   :", ids)
print("Decode:", decoding(ids))

### 6. Dataset e Dataloader

In [ ]:
class TranslationDataset(Dataset):

    def __init__(
        self,
        samples, # Lista de frases em português e inglês
        tokenizer):
        
        self.samples = samples
        self.src_samples, self.tgt_samples = zip(*samples)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        src_sample = self.src_samples[idx]
        tgt_sample = self.tgt_samples[idx]

        src_ids = (self.tokenizer(src_sample) )
            
        tgt_ids = (self.tokenizer(tgt_sample) )

        return (
            torch.tensor(src_ids, dtype=torch.long),
            torch.tensor(tgt_ids, dtype=torch.long)
        )

In [ ]:
# Função que inclui o padding nas amostras
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):

    src_batch, tgt_batch = zip(*batch)

    src_batch = pad_sequence(
        src_batch,
        batch_first=True,
        padding_value=pad_id
    )

    tgt_batch = pad_sequence(
        tgt_batch,
        batch_first=True,
        padding_value=pad_id
    )

    return src_batch, tgt_batch

In [ ]:
# Cria o Dataset e o Dataloader de treinamento

from torch.utils.data import DataLoader
train_pairs = list(zip(train_pt, train_en))

train_dataset = TranslationDataset(train_pairs, encoding) # cria o dataset de treinamento

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
) # cria o dataloader de treinamento

In [ ]:
len(train_dataset)

In [ ]:
len(train_loader)

In [ ]:
one_batch_src, one_batch_tgt = next(iter(train_loader))

print(one_batch_src[0])
print(one_batch_tgt[0])

In [ ]:
print(decoding(one_batch_src[0].tolist()))
print(100*'-')
print(decoding(one_batch_tgt[0].tolist()))

In [ ]:
# Dataload de validação
valid_pairs = list(zip(valid_pt, valid_en))

valid_dataset = TranslationDataset(
    valid_pairs,
    encoding
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)


In [ ]:
train_dataset[0]

In [ ]:
src_lengths = [len(item[0]) for item in train_dataset] # obtem os tamanhos das amostras src
tgt_lengths = [len(item[1]) for item in train_dataset] # obtem os tamanhos das amostras tgt

print(max(src_lengths)) # obtem o tamanho da maior amostra src
print(max(tgt_lengths)) # obtem o tamanho da maior amostra tgt

print(sum(src_lengths) / len(src_lengths)) # calcula o tamanho médio das amostras src
print(sum(tgt_lengths) / len(tgt_lengths)) # calcula o tamanho médio das amostras tgt

In [ ]:
src, tgt = next(iter(train_loader))

print(src.shape)  # (BATCH_SIZE,xx)
print(tgt.shape)  # (BATCH_SIZE,yy)


In [ ]:
src[0]

In [ ]:
print(decoding(src[0].tolist()))
print(decoding(tgt[0].tolist()))


### 7. Embedding

In [ ]:
import torch
import torch.nn as nn



embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)

# Testando
x = torch.randint(0, VOCAB_SIZE, (32, 20))

emb = embedding(x)

print(emb.shape)


### 8. Positional Encoder

In [ ]:
# =====================================================
# Positional Encoding
# =====================================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(
            0, max_len, dtype=torch.float
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0, d_model, 2,
                dtype=torch.float
            ) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]


In [ ]:
pos = PositionalEncoding(D_MODEL)

x = pos(emb)

### 9. Scaled Dot Product Attention

In [ ]:
# =====================================================
# Scaled Dot Product Attention
# =====================================================

class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, q, k, v, mask=None):

        d_k = q.size(-1)

        scores = torch.matmul(
            q, k.transpose(-2, -1)
        ) / math.sqrt(d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attention = F.softmax(scores, dim=-1)

        output = torch.matmul(attention, v)

        #return output, attention
        return output

### 10. Multi-Head Attention

In [ ]:
# =====================================================
# Multi Head Attention
# =====================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        self.w_o = nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention()

    def split_heads(self, x):

        batch_size = x.size(0)

        x = x.view(
            batch_size,
            -1,
            self.num_heads,
            self.d_k
        )

        return x.transpose(1, 2)

    def combine_heads(self, x):

        batch_size = x.size(0)

        x = x.transpose(1, 2).contiguous()

        return x.view(
            batch_size,
            -1,
            self.d_model
        )

    def forward(self, q, k, v, mask=None):

        q = self.split_heads(self.w_q(q))
        k = self.split_heads(self.w_k(k))
        v = self.split_heads(self.w_v(v))

        #output, attn = self.attention(
        output = self.attention(
            q,
            k,
            v,
            mask
        )

        output = self.combine_heads(output)

        output = self.w_o(output)

        return output


### 11. Feed Forward Network

In [ ]:
# =====================================================
# Feed Forward Network
# =====================================================

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff=2048, dropout=0.1):
        super().__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

### 12. Encoder Layer

In [ ]:
# =====================================================
# Encoder Layer
# =====================================================

class EncoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.self_attn = MultiHeadAttention(
            d_model,
            num_heads
        )

        self.ffn = PositionwiseFeedForward(
            d_model,
            d_ff,
            dropout
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):

        attn_output = self.self_attn(
            x, x, x, src_mask
        )

        x = self.norm1(
            x + self.dropout(attn_output)
        )

        ff_output = self.ffn(x)

        x = self.norm2(
            x + self.dropout(ff_output)
        )

        return x

### 13. Decoder Layer

In [ ]:
# =====================================================
# Decoder Layer
# =====================================================

class DecoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.self_attn = MultiHeadAttention(
            d_model,
            num_heads
        )

        self.cross_attn = MultiHeadAttention(
            d_model,
            num_heads
        )

        self.ffn = PositionwiseFeedForward(
            d_model,
            d_ff,
            dropout
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x,
        encoder_output,
        src_mask,
        tgt_mask
    ):

        attn = self.self_attn(
            x, x, x, tgt_mask
        )

        x = self.norm1(
            x + self.dropout(attn)
        )

        attn = self.cross_attn(
            x,
            encoder_output,
            encoder_output,
            src_mask
        )

        x = self.norm2(
            x + self.dropout(attn)
        )

        ff = self.ffn(x)

        x = self.norm3(
            x + self.dropout(ff)
        )

        return x


### 14. Encoder

In [ ]:
# =====================================================
# Encoder
# =====================================================

class Encoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        num_layers,
        num_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.pos_encoding = PositionalEncoding(
            d_model
        )

        self.layers = nn.ModuleList(
            [
                EncoderLayer(
                    d_model,
                    num_heads,
                    d_ff,
                    dropout
                )
                for _ in range(num_layers)
            ]
        )

        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def forward(self, src, src_mask):

        x = self.embedding(src) * math.sqrt(
            self.d_model
        )

        x = self.pos_encoding(x)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, src_mask)

        return x


### 15. Decoder

In [ ]:
# =====================================================
# Decoder
# =====================================================

class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        num_layers,
        num_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.pos_encoding = PositionalEncoding(
            d_model
        )

        self.layers = nn.ModuleList(
            [
                DecoderLayer(
                    d_model,
                    num_heads,
                    d_ff,
                    dropout
                )
                for _ in range(num_layers)
            ]
        )

        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def forward(
        self,
        tgt,
        encoder_output,
        src_mask,
        tgt_mask
    ):

        x = self.embedding(tgt) * math.sqrt(
            self.d_model
        )

        x = self.pos_encoding(x)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(
                x,
                encoder_output,
                src_mask,
                tgt_mask
            )

        return x

### 16. Transformer

In [ ]:
# =====================================================
# Transformer Completo
# =====================================================

class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=512,
        num_layers=6,
        num_heads=8,
        d_ff=2048,
        dropout=0.1
    ):
        super().__init__()

        self.encoder = Encoder(
            src_vocab_size,
            d_model,
            num_layers,
            num_heads,
            d_ff,
            dropout
        )

        self.decoder = Decoder(
            tgt_vocab_size,
            d_model,
            num_layers,
            num_heads,
            d_ff,
            dropout
        )

        self.fc_out = nn.Linear(
            d_model,
            tgt_vocab_size
        )

    def forward(
        self,
        src,
        tgt,
        src_mask,
        tgt_mask
    ):

        encoder_output = self.encoder(
            src,
            src_mask
        )

        decoder_output = self.decoder(
            tgt,
            encoder_output,
            src_mask,
            tgt_mask
        )

        output = self.fc_out(
            decoder_output
        )

        return output


### 17. Máscaras

In [ ]:
# =====================================================
# Máscaras
# =====================================================

def create_padding_mask(seq, pad_idx=pad_id):
    return (seq != pad_idx).unsqueeze(1).unsqueeze(2)


def create_causal_mask(size):

    mask = torch.tril(
        torch.ones(size, size)
    )

    return mask.bool().unsqueeze(0).unsqueeze(1)

### 18. Aplicação

In [ ]:
src_vocab_size = VOCAB_SIZE
tgt_vocab_size = VOCAB_SIZE

model = Transformer(
    src_vocab_size,
    tgt_vocab_size
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.98),
    eps=1e-9
)



In [ ]:
#PAD_IDX = tokenizer.token_to_id("[PAD]")

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)


In [ ]:
# Testando o modelo

src = torch.randint(
    1,
    src_vocab_size,
    (4, 20)
).to(device)

tgt = torch.randint(
    1,
    tgt_vocab_size,
    (4, 15)
).to(device)

src_mask = create_padding_mask(src).to(device)

tgt_input = tgt[:, :-1].to(device)

tgt_output = tgt[:, 1:].to(device)

tgt_mask = (
    create_padding_mask(tgt_input).to(device)
    & create_causal_mask(tgt_input.size(1)).to(device)
)

output = model(
    src,
    tgt_input,
    src_mask,
    tgt_mask
)

loss = criterion(
    output.reshape(-1, output.size(-1)),
    tgt_output.reshape(-1)
)
print(output.shape)
# (4, 14, VOCAB_SIZE)

### 19. Loop de treinamento

In [ ]:
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Transformer(
    src_vocab_size,
    tgt_vocab_size,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_ENCODER_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.98),
    eps=1e-9
)

In [ ]:
import gc
import time
import torch
from tqdm.auto import tqdm

def train_epoch(model, dataloader, optimizer, criterion, device, pad_id, epoch):
    model.train()

    total_loss = 0.0
    start = time.time()

    for src, tgt in tqdm(dataloader, desc=f"[TREINO] Epoch {epoch+1}"):

        src = src.to(device)
        tgt = tgt.to(device)

        # entrada do decoder
        tgt_input = torch.full(
            (tgt.size(0), tgt.size(1)),
            pad_id,
            device=device
        )
        tgt_input[:, 1:] = tgt[:, :-1]

        tgt_output = tgt

        # máscaras
        src_mask = create_padding_mask(src, pad_id).to(device)

        tgt_mask = (
            create_padding_mask(tgt_input, pad_id)
            &
            create_causal_mask(tgt_input.size(1)).to(device)
        )

        optimizer.zero_grad()

        output = model(src, tgt_input, src_mask, tgt_mask)

        loss = criterion(
            output.reshape(-1, output.size(-1)),
            tgt_output.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

    del output
    gc.collect()
    torch.cuda.empty_cache()

    avg_loss = total_loss / len(dataloader)
    elapsed = time.time() - start

    return avg_loss, elapsed

In [ ]:
def validate_epoch(model, dataloader, criterion, device, pad_id):
    model.eval()

    total_loss = 0.0

    with torch.no_grad():
        for src, tgt in tqdm(dataloader, desc="[VALIDAÇÃO]"):

            src = src.to(device)
            tgt = tgt.to(device)

            tgt_input = torch.full(
                (tgt.size(0), tgt.size(1)),
                pad_id,
                device=device
            )
            tgt_input[:, 1:] = tgt[:, :-1]

            tgt_output = tgt

            src_mask = create_padding_mask(src, pad_id).to(device)

            tgt_mask = (
                create_padding_mask(tgt_input, pad_id)
                &
                create_causal_mask(tgt_input.size(1)).to(device)
            )

            output = model(src, tgt_input, src_mask, tgt_mask)

            loss = criterion(
                output.reshape(-1, output.size(-1)),
                tgt_output.reshape(-1)
            )

            total_loss += loss.item()

            del output

    avg_loss = total_loss / len(dataloader)
    return avg_loss

In [ ]:
NUM_EPOCHS = 30

train_losses = []
val_losses = []

best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):

    train_loss, elapsed = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device,
        pad_id,
        epoch
    )

    val_loss = validate_epoch(
        model,
        valid_loader,
        criterion,
        device,
        pad_id
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val   Loss: {val_loss:.4f}")
    print(f"Tempo: {elapsed:.1f} s")

    # salva o melhor modelo
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss
        }, 'best_transformer.pt')

        print("\u2714 Melhor modelo salvo!")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Treino x Validação')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
NUM_EPOCHS = 20

for epoch in range(NUM_EPOCHS):

    loss = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device,
        epoch
    )

    print(f"Epoch {epoch+1:02d} | Loss = {loss:.4f}\n")

### 20. Salvamento do Modelo

In [ ]:
torch.save(
    model.state_dict(),
    "translator.pt"
)

In [ ]:
# Load
# model.load_state_dict(
#     torch.load("translator.pt")
# )

### 21. Inferência

In [ ]:
@torch.no_grad()
def translate(sentence, max_len=None):
    model.eval()

    src_ids = encoding(sentence)

    src = torch.tensor(
        [src_ids],
        dtype=torch.long,
        device=device
    )

    src_mask = create_padding_mask(src)

    encoder_output = model.encoder(src, src_mask)

    tgt = torch.tensor(
        [[pad_id]],
        dtype=torch.long,
        device=device
    )

    if max_len is None:
        max_len = min(len(src_ids) + 30, 128)

    for _ in range(max_len):

        tgt_mask = (
            create_padding_mask(tgt)
            &
            create_causal_mask(tgt.size(1)).to(device)
        )

        decoder_output = model.decoder(
            tgt,
            encoder_output,
            src_mask,
            tgt_mask
        )

        logits = model.fc_out(decoder_output)

        next_token = logits[:, -1].argmax(dim=-1)

        tgt = torch.cat(
            (tgt, next_token.unsqueeze(1)),
            dim=1
        )

        if next_token.item() == eos_id:
            break

    tokens = [
        t
        for t in tgt[0].tolist()
        if t not in (eos_id, pad_id)
    ]

    return decoding(tokens)

In [ ]:
sample = random.sample(train_pt, 1)[0]
print(sample)
translate(sample)